In [3]:
import duckdb
import os
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv
import os
import plotly.express as px
import plotly.graph_objects as go
import json

load_dotenv()


True

In [4]:
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


folder_path='../data/AdventureWorks'
db_path="../data/AdventureWorks.duckdb"

# conn = duckdb.connect(db_path)
conn = duckdb.connect(":memory:")

folder = Path(folder_path)

for csv_file in folder.glob("*.csv"):

    table_name = csv_file.stem  # file name without .csv

    query = f"""
    CREATE OR REPLACE TABLE {table_name} AS
    SELECT * FROM read_csv_auto('{csv_file}');
    """

    conn.execute(query)

    print(f"Created table: {table_name}")

conn.close()

Created table: WorkOrder
Created table: TransactionHistory
Created table: WorkOrderRouting
Created table: PurchaseOrderDetail
Created table: Product


In [3]:
db_path="../data/AdventureWorks.duckdb"
# conn = duckdb.connect(db_path)
conn = duckdb.connect(":memory:")

# Inspect all tables
tables = conn.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetchdf()

print(f"Found {len(tables)} tables:")
for t in tables["table_name"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t} ({count} rows)")

Found 5 tables:
  Product (504 rows)
  PurchaseOrderDetail (8845 rows)
  TransactionHistory (65535 rows)
  WorkOrder (65535 rows)
  WorkOrderRouting (59832 rows)


In [81]:
from pydantic import BaseModel, Field
from typing import Literal

# ── Model 1: Tool Input ──────────────────────────────────────────
# Validates what the agent sends TO the tool before it hits DuckDB
class RunSqlInput(BaseModel):
    sql: str = Field(..., description="Valid DuckDB SQL query")

# ── Model 2: Agent Final Answer ──────────────────────────────────
# Enforces the shape of what the agent returns TO you
class AgentAnswer(BaseModel):
    answer: str = Field(..., description="Plain English explanation of the result")
    sql_used: str = Field(..., description="The SQL that produced the final result")
    chart_type: Literal["bar", "line", "pie", "scatter", "histogram", "box", "none"]
    plotly_code: str = Field(..., description="Complete plotly figure code using df and fig variables")

print("✅ Models defined")
print(f"   RunSqlInput fields : {list(RunSqlInput.model_fields.keys())}")
print(f"   AgentAnswer fields : {list(AgentAnswer.model_fields.keys())}")


✅ Models defined
   RunSqlInput fields : ['sql']
   AgentAnswer fields : ['answer', 'sql_used', 'chart_type', 'plotly_code']


In [5]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_tables",
            "description": "List all available tables in the database. Always call this first.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "Get schema and sample rows for a specific table. Call this for each relevant table before writing SQL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "table_name": {
                        "type": "string",
                        "description": "Name of the table to inspect"
                    }
                },
                "required": ["table_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": "Run a DuckDB SQL query across one or more tables. Use JOINs when needed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "Valid DuckDB SQL. No LIMIT on aggregations. LIMIT 100 on raw rows."
                    }
                },
                "required": ["sql"]
            }
        }
    }
]

In [8]:
def execute_tool(tool_name: str, tool_args: dict) -> str:

    if tool_name == "list_tables":
        tables = conn.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
        """).fetchdf()
        return tables["table_name"].tolist().__str__()

    elif tool_name == "get_schema":
        table = tool_args.get("table_name")

        # schema for specific table
        cols = conn.execute(f"DESCRIBE {table}").fetchdf()
        count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

        # sample 3 rows — helps agent understand join keys
        sample = conn.execute(f"SELECT * FROM {table} LIMIT 3").fetchdf()

        result = f"TABLE: {table} ({count} rows)\n"
        result += cols[["column_name", "column_type"]].to_string(index=False)
        result += f"\n\nSAMPLE:\n{sample.to_string(index=False)}"
        return result

    elif tool_name == "run_sql":
        try:
            validated = RunSqlInput(**tool_args)
            last_df = conn.execute(validated.sql).fetchdf()
            return last_df.to_string(index=False), last_df
        except Exception as e:
            return f"SQL_ERROR: {str(e)}", None

    return f"UNKNOWN_TOOL: {tool_name}"

In [9]:
MAX_RETRIES = 3
AGENT_MODEL = "gpt-5-mini"
FINAL_MODEL = "gpt-5.4"

# SYSTEM_PROMPT = """You are a data analyst agent with access to a multi-table database.
#
# Follow this exact process:
# 1. Call list_tables to discover all available tables
# 2. Call get_schema for each relevant table
# 3. Identify join keys from column names and sample data
# 4. Write SQL with proper JOINs to answer the question
# 5. Repeat step 4 if needed
#
# SQL rules:
# - Use DuckDB SQL syntax
# - Always qualify column names with table name e.g. orders.customer_id
# - No LIMIT on GROUP BY queries
# - LIMIT 100 on raw row queries
# - Prefer LEFT JOIN unless INNER JOIN is clearly needed
#
# For plotly_code:
# - df is already loaded with your last query result
# - create a figure called fig
# - do NOT call fig.show()
# """

SYSTEM_PROMPT = """You are a data analyst agent with access to a multi-table database.

TOOL USAGE RULES:
- list_tables  → only call ONCE per conversation, skip if already in history
- get_schema   → only for tables directly needed, max 2-3 per question, skip if already fetched in this conversation
- run_sql      → required for ANY question asking for data, numbers, or graphs

QUESTION TYPES:
- "brief", "describe", "what tables" → list_tables only, no SQL
- "show", "graph", "how many", "top", "compare", "who", "which" → MUST run SQL

SQL rules:
- No LIMIT on GROUP BY queries
- LIMIT 100 on raw rows
- Always qualify column names with table name
- Prefer LEFT JOIN unless INNER JOIN clearly needed

For plotly_code:
- df is already loaded, create figure called fig, do NOT call fig.show()
"""


def run_agent(question: str) -> AgentAnswer:
    global conversation_history

    conversation_history.append({"role": "user", "content": question})
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history

    print(f"\nQ: {question}")
    print("-" * 40)

    step = 0
    retries = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        conversation_history.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "list_tables":
                print(f"Step {step}: list_tables()")
                result = execute_tool("list_tables", {})

            elif tc.function.name == "get_schema":
                print(f"Step {step}: get_schema({args.get('table_name')})")
                result = execute_tool("get_schema", args)

            elif tc.function.name == "run_sql":
                print(f"Step {step}: run_sql()")

                # ── Pydantic validation ───────────────────────────
                try:
                    validated = RunSqlInput(**args)
                except Exception as e:
                    result = f"VALIDATION_ERROR: {str(e)}"
                    print(f"  Validation failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                    messages.append(tool_msg)
                    conversation_history.append(tool_msg)
                    continue

                print(f"  {validated.sql}")

                # ── SQL execution ─────────────────────────────────
                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                    retries = 0
                except Exception as e:
                    result = f"SQL_ERROR: {str(e)}"
                    print(f"  SQL failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    print(f"  Retry {retries}/{MAX_RETRIES} — feeding error back to agent")

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            conversation_history.append(tool_msg)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []

        # only mention df if we actually have one
        df_context = (
            f"The final dataframe has these exact columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer based on schema information only. Set chart_type to none."
        )

        final = client.beta.chat.completions.parse(
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed

    except Exception as e:
        print(f"Final answer error: {e}")
        return None

    conversation_history.append({"role": "assistant", "content": result.answer})

    # ── Display ───────────────────────────────────────────────────
    print(f"\nAnswer: {result.answer}")
    print(f"Chart:  {result.chart_type}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            print(f"Chart error: {e}")

    return result

In [10]:
def reset_memory():
    global conversation_history
    conversation_history = []
    print("Memory cleared.")

reset_memory()

Memory cleared.


In [11]:
run_agent("Hi, give me a brief about the data")


Q: Hi, give me a brief about the data
----------------------------------------
Step 1: list_tables()

Answer: The database contains five tables that appear to cover products, purchasing, inventory transactions, and manufacturing operations:

- Product: likely the master list of products and their attributes.
- PurchaseOrderDetail: likely purchase-order line items tied to products, quantities, and costs.
- TransactionHistory: likely records historical inventory or product-related transactions over time.
- WorkOrder: likely stores manufacturing or production work orders for products.
- WorkOrderRouting: likely stores the operational steps or routing details associated with work orders.

Based on table names alone, this looks like an operational dataset for supply chain and manufacturing analysis.
Chart:  none


AgentAnswer(answer='The database contains five tables that appear to cover products, purchasing, inventory transactions, and manufacturing operations:\n\n- Product: likely the master list of products and their attributes.\n- PurchaseOrderDetail: likely purchase-order line items tied to products, quantities, and costs.\n- TransactionHistory: likely records historical inventory or product-related transactions over time.\n- WorkOrder: likely stores manufacturing or production work orders for products.\n- WorkOrderRouting: likely stores the operational steps or routing details associated with work orders.\n\nBased on table names alone, this looks like an operational dataset for supply chain and manufacturing analysis.', sql_used='', chart_type='none', plotly_code='')

In [12]:
run_agent("give me 3 bestselling products")


Q: give me 3 bestselling products
----------------------------------------
Step 1: get_schema(Product)
Step 2: get_schema(TransactionHistory)
Step 3: run_sql()
  SELECT TransactionHistory.TransactionType, COUNT(*) AS cnt, SUM(TransactionHistory.Quantity) AS total_quantity
FROM TransactionHistory
GROUP BY TransactionHistory.TransactionType
ORDER BY total_quantity DESC;
Step 4: run_sql()
  SELECT Product.ProductID, Product.Name, SUM(TransactionHistory.Quantity) AS total_sold
FROM TransactionHistory
JOIN Product ON TransactionHistory.ProductID = Product.ProductID
WHERE TransactionHistory.TransactionType = 'S'
GROUP BY Product.ProductID, Product.Name
ORDER BY total_sold DESC
LIMIT 3;

Answer: Here are the 3 bestselling products by total quantity sold:
1. Water Bottle - 30 oz. (ProductID 870) — 3,428 units
2. AWC Logo Cap (ProductID 712) — 2,548 units
3. Classic Vest, S (ProductID 864) — 2,274 units
Chart:  bar


AgentAnswer(answer='Here are the 3 bestselling products by total quantity sold:\n1. Water Bottle - 30 oz. (ProductID 870) — 3,428 units\n2. AWC Logo Cap (ProductID 712) — 2,548 units\n3. Classic Vest, S (ProductID 864) — 2,274 units', sql_used="SELECT Product.ProductID, Product.Name, SUM(TransactionHistory.Quantity) AS total_sold\nFROM TransactionHistory\nLEFT JOIN Product ON TransactionHistory.ProductID = Product.ProductID\nWHERE TransactionHistory.TransactionType = 'S'\nGROUP BY Product.ProductID, Product.Name\nORDER BY total_sold DESC\nLIMIT 3;", chart_type='bar', plotly_code="import plotly.express as px\nfig = px.bar(df, x='Name', y='total_sold', text='total_sold', hover_data=['ProductID'], title='Top 3 Bestselling Products')\nfig.update_traces(textposition='outside')\nfig.update_layout(xaxis_title='Product', yaxis_title='Units Sold')")

In [15]:
run_agent("give me month by month  sales number vs revenue")


Q: give me month by month  sales number vs revenue
----------------------------------------
Step 1: run_sql()
  SELECT strftime('%Y-%m', TransactionHistory.TransactionDate) AS year_month,
       SUM(TransactionHistory.Quantity) AS total_quantity,
       SUM(TransactionHistory.Quantity * CAST(REPLACE(REPLACE(TRIM(COALESCE(Product.ListPrice,'$0.00')),'$',''),',','') AS DOUBLE)) AS total_revenue
FROM TransactionHistory
LEFT JOIN Product ON TransactionHistory.ProductID = Product.ProductID
WHERE TransactionHistory.TransactionType = 'S'
GROUP BY year_month
ORDER BY year_month;

Answer: Month-by-month sales comparison:
- 2013-07: 14,880 units, $7,356,830.33
- 2013-08: 11,548 units, $4,909,036.38
- 2013-09: 14,576 units, $6,890,589.82
- 2013-10: 14,984 units, $7,164,610.48
- 2013-11: 9,667 units, $4,430,540.45
- 2013-12: 11,049 units, $5,889,090.92
- 2014-01: 11,425 units, $6,103,933.92
Chart:  line


AgentAnswer(answer='Month-by-month sales comparison:\n- 2013-07: 14,880 units, $7,356,830.33\n- 2013-08: 11,548 units, $4,909,036.38\n- 2013-09: 14,576 units, $6,890,589.82\n- 2013-10: 14,984 units, $7,164,610.48\n- 2013-11: 9,667 units, $4,430,540.45\n- 2013-12: 11,049 units, $5,889,090.92\n- 2014-01: 11,425 units, $6,103,933.92', sql_used="SELECT strftime('%Y-%m', TransactionHistory.TransactionDate) AS year_month,\n       SUM(TransactionHistory.Quantity) AS total_quantity,\n       SUM(TransactionHistory.Quantity * CAST(REPLACE(REPLACE(TRIM(COALESCE(Product.ListPrice,'$0.00')),'$',''),',','') AS DOUBLE)) AS total_revenue\nFROM TransactionHistory\nLEFT JOIN Product ON TransactionHistory.ProductID = Product.ProductID\nWHERE TransactionHistory.TransactionType = 'S'\nGROUP BY year_month\nORDER BY year_month;", chart_type='line', plotly_code="import plotly.graph_objects as go\n\nfig = go.Figure()\nfig.add_trace(go.Scatter(x=df['year_month'], y=df['total_quantity'], mode='lines+markers', na

# memory

In [38]:
import tiktoken

class MemoryManager:

    def __init__(self, max_turns: int = 6):
        self.max_turns = max_turns
        self.history = []       # user questions + agent answers only
        self.summary = None     # single rolling summary of older turns

    def add_question(self, question: str):
        self.history.append({"role": "user", "content": question})

    def add_answer(self, answer: str):
        self.history.append({"role": "assistant", "content": answer})
        self._maybe_summarise()

    def _maybe_summarise(self):
        """Triggered after every answer. Summarises when history exceeds max_turns."""
        if len(self.history) <= self.max_turns * 2:
            return

        # split
        keep_from = -(self.max_turns * 2)
        to_summarise = self.history[:keep_from]
        self.history = self.history[keep_from:]

        # include existing summary if present
        prior = f"Previous summary:\n{self.summary}\n\nNew turns:\n" if self.summary else ""
        text = prior + "\n".join(
            f"{m['role']}: {m['content']}"
            for m in to_summarise
            if isinstance(m.get("content"), str)
        )

        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=[{
                "role": "user",
                "content": f"Summarise this conversation. Keep key questions, findings, numbers, insights. 4-5 sentences.\n\n{text}"
            }]
        )

        self.summary = response.choices[0].message.content
        print(f"  [memory] summarised — keeping last {self.max_turns} turns")

    def get_messages(self, system_prompt: str) -> list:
        system = [{"role": "system", "content": system_prompt}]

        summary_msg = []
        if self.summary:
            summary_msg = [{"role": "system", "content": f"[Earlier context]: {self.summary}"}]

        return system + summary_msg + self.history

    def reset(self):
        self.history = []
        self.summary = None
        print("  [memory] reset.")

    def stats(self):
        print(f"  [memory] turns: {len(self.history)//2}/{self.max_turns} | summary: {'yes' if self.summary else 'no'}")

In [39]:
memory = MemoryManager(max_turns=6)


In [12]:
MAX_RETRIES = 3
AGENT_MODEL = "gpt-5-mini"
FINAL_MODEL = "gpt-5.4"

# SYSTEM_PROMPT = """You are a data analyst agent with access to a multi-table database.
#
# Follow this exact process:
# 1. Call list_tables to discover all available tables
# 2. Call get_schema for each relevant table
# 3. Identify join keys from column names and sample data
# 4. Write SQL with proper JOINs to answer the question
# 5. Repeat step 4 if needed
#
# SQL rules:
# - Use DuckDB SQL syntax
# - Always qualify column names with table name e.g. orders.customer_id
# - No LIMIT on GROUP BY queries
# - LIMIT 100 on raw row queries
# - Prefer LEFT JOIN unless INNER JOIN is clearly needed
#
# For plotly_code:
# - df is already loaded with your last query result
# - create a figure called fig
# - do NOT call fig.show()
# """

SYSTEM_PROMPT = """You are a data analyst agent with access to a multi-table database.

TOOL USAGE RULES:
- list_tables  → only call ONCE per conversation, skip if already in history
- get_schema   → only for tables directly needed, max 2-3 per question, skip if already fetched in this conversation
- run_sql      → required for ANY question asking for data, numbers, or graphs

QUESTION TYPES:
- "brief", "describe", "what tables" → list_tables only, no SQL
- "show", "graph", "how many", "top", "compare", "who", "which" → MUST run SQL

SQL rules:
- No LIMIT on GROUP BY queries
- LIMIT 100 on raw rows
- Always qualify column names with table name
- Prefer LEFT JOIN unless INNER JOIN clearly needed

For plotly_code:
- df is already loaded, create figure called fig, do NOT call fig.show()
"""


def run_agent_with_memory(question: str) -> AgentAnswer:

    memory.add({"role": "user", "content": question})
    messages = memory.get_messages(SYSTEM_PROMPT)

    print(f"\nQ: {question}")
    print("-" * 40)

    step = 0
    retries = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        memory.add(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "list_tables":
                print(f"Step {step}: list_tables()")
                result = execute_tool("list_tables", {})

            elif tc.function.name == "get_schema":
                print(f"Step {step}: get_schema({args.get('table_name')})")
                result = execute_tool("get_schema", args)

            elif tc.function.name == "run_sql":
                print(f"Step {step}: run_sql()")

                # ── Pydantic validation ───────────────────────────
                try:
                    validated = RunSqlInput(**args)
                except Exception as e:
                    result = f"VALIDATION_ERROR: {str(e)}"
                    print(f"  Validation failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                    messages.append(tool_msg)
                    memory.add(tool_msg)
                    continue

                print(f"  {validated.sql}")

                # ── SQL execution ─────────────────────────────────
                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                    retries = 0
                except Exception as e:
                    result = f"SQL_ERROR: {str(e)}"
                    print(f"  SQL failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    print(f"  Retry {retries}/{MAX_RETRIES} — feeding error back to agent")

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            memory.add(tool_msg)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []

        # only mention df if we actually have one
        df_context = (
            f"The final dataframe has these exact columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer based on schema information only. Set chart_type to none."
        )

        final = client.beta.chat.completions.parse(
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed

    except Exception as e:
        print(f"Final answer error: {e}")
        return None

    memory.add({"role": "assistant", "content": result.answer})

    # ── Display ───────────────────────────────────────────────────
    print(f"\nAnswer: {result.answer}")
    print(f"Chart:  {result.chart_type}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            print(f"Chart error: {e}")

    return result

In [37]:
run_agent_with_memory('show me monthly sales graph')


Q: show me monthly sales graph
----------------------------------------
Step 1: list_tables()
Step 2: get_schema(TransactionHistory)
Step 3: run_sql()
  SELECT TransactionType, COUNT(*) AS cnt
FROM TransactionHistory
GROUP BY TransactionType
ORDER BY cnt DESC;
Step 4: run_sql()
  SELECT strftime(TransactionHistory.TransactionDate, '%Y-%m') AS year_month,
       SUM(CAST(REPLACE(REPLACE(TransactionHistory.ActualCost,'$',''),',','') AS DOUBLE)) AS revenue,
       SUM(TransactionHistory.Quantity) AS units
FROM TransactionHistory
WHERE TransactionHistory.TransactionType = 'S'
GROUP BY year_month
ORDER BY year_month;

Answer: Monthly sales are available from Jul 2013 to Jan 2014. Revenue rises overall from about $1.68M in 2013-07 to a peak of about $2.80M in 2014-01. Units sold fluctuate, with the highest level in 2013-10 at 14,984 units.
Chart:  line


AgentAnswer(answer='Monthly sales are available from Jul 2013 to Jan 2014. Revenue rises overall from about $1.68M in 2013-07 to a peak of about $2.80M in 2014-01. Units sold fluctuate, with the highest level in 2013-10 at 14,984 units.', sql_used="SELECT strftime(TransactionHistory.TransactionDate, '%Y-%m') AS year_month,\n       SUM(CAST(REPLACE(REPLACE(TransactionHistory.ActualCost,'$',''),',','') AS DOUBLE)) AS revenue,\n       SUM(TransactionHistory.Quantity) AS units\nFROM TransactionHistory\nWHERE TransactionHistory.TransactionType = 'S'\nGROUP BY year_month\nORDER BY year_month;", chart_type='line', plotly_code="import pandas as pd\nimport plotly.graph_objects as go\n\ndf['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')\n\nfig = go.Figure()\nfig.add_trace(go.Scatter(\n    x=df['year_month'],\n    y=df['revenue'],\n    mode='lines+markers',\n    name='Revenue',\n    line=dict(color='steelblue', width=3),\n    hovertemplate='%{x|%b %Y}<br>Revenue: $%{y:,.2f}<extra

In [39]:
memory.history

[{'role': 'user', 'content': 'show me monthly sales graph'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_uohVE6b54J2DhSL8ss7tot4X', function=Function(arguments='{}', name='list_tables'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_uohVE6b54J2DhSL8ss7tot4X',
  'content': "['Product', 'PurchaseOrderDetail', 'TransactionHistory', 'WorkOrder', 'WorkOrderRouting']"},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_DFMvttjuJ40h5mBGRPRRM3ph', function=Function(arguments='{"table_name":"TransactionHistory"}', name='get_schema'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_DFMvttjuJ40h5mBGRPRRM3ph',
  'content': 'TABLE: TransactionHistory (65535 rows)\n         column_name column_type\n       Transaction

# Logging

In [115]:
pwd

'/Users/anshulgautam/Projects/datavis_agent/notebooks'

In [117]:
import os
from loguru import logger

os.makedirs("../logs", exist_ok=True)

# ── Remove default handler, configure fresh ───────────────────────
logger.remove()

# ── Console — INFO and above ──────────────────────────────────────
logger.add(
    sink=lambda msg: print(msg, end=""),
    level="INFO",
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}"
)

# ── File — DEBUG and above (everything) ───────────────────────────
logger.add(
    sink="../logs/agent_{time:YYYY-MM-DD}.log",
    level="DEBUG",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}",
    rotation="1 day",       # new file every day
    retention="7 days",     # keep last 7 days only
    compression="zip"       # compress old logs
)

6

In [22]:
logger.debug("tool args received")        # file only
logger.info("agent started")              # console + file
logger.warning("retry attempt 1/3")       # console + file
logger.error("SQL failed")

10:14:39 | INFO     | agent started
10:14:39 | WARNING  | retry attempt 1/3
10:14:39 | ERROR    | SQL failed


In [15]:
def run_agent_with_error_handling(question: str) -> AgentAnswer:

    memory.add({"role": "user", "content": question})
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    logger.debug(f"messages in context: {len(messages)}")
    memory.stats()

    step = 0
    retries = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        memory.add(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "list_tables":
                logger.info(f"step {step}: list_tables()")
                result = execute_tool("list_tables", {})

            elif tc.function.name == "get_schema":
                logger.info(f"step {step}: get_schema({args.get('table_name')})")
                result = execute_tool("get_schema", args)

            elif tc.function.name == "run_sql":
                logger.info(f"step {step}: run_sql()")

                try:
                    validated = RunSqlInput(**args)
                except Exception as e:
                    logger.warning(f"validation failed: {e}")
                    retries += 1
                    result = f"VALIDATION_ERROR: {str(e)}"
                    if retries >= MAX_RETRIES:
                        logger.error("max retries reached. stopping.")
                        return None
                    tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                    messages.append(tool_msg)
                    memory.add(tool_msg)
                    continue

                logger.info(f"sql: {validated.sql}")

                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                    logger.debug(f"rows returned: {len(last_df)}")
                    retries = 0
                except Exception as e:
                    logger.error(f"sql failed: {e}")
                    result = f"SQL_ERROR: {str(e)}"
                    retries += 1
                    if retries >= MAX_RETRIES:
                        logger.error("max retries reached. stopping.")
                        return None
                    logger.warning(f"retry {retries}/{MAX_RETRIES}")

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            memory.add(tool_msg)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from schema only. Set chart_type to none."
        )

        final = client.beta.chat.completions.parse(
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        return None

    memory.add({"role": "assistant", "content": result.answer})

    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart: {result.chart_type}")
    logger.debug(f"sql used: {result.sql_used}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [16]:
run_agent_with_error_handling('show me the graphs of best seler product for each month')

15:57:46 | INFO     | question: show me the graphs of best seler product for each month
  [memory] turns: 1 | summaries: 0 | tokens: 12/6000
15:57:50 | INFO     | step 1: list_tables()
15:57:52 | INFO     | step 2: get_schema(PurchaseOrderDetail)
15:57:54 | INFO     | step 3: get_schema(Product)
15:57:58 | INFO     | step 4: get_schema(TransactionHistory)
15:58:04 | INFO     | step 5: run_sql()
15:58:04 | INFO     | sql: SELECT TransactionHistory.TransactionType, COUNT(*) AS cnt
FROM TransactionHistory
GROUP BY TransactionHistory.TransactionType
ORDER BY cnt DESC;
15:58:16 | INFO     | step 6: run_sql()
15:58:16 | INFO     | sql: WITH monthly AS (
  SELECT
    strftime('%Y-%m', TransactionHistory.TransactionDate) AS month,
    TransactionHistory.ProductID,
    SUM(TransactionHistory.Quantity) AS total_qty
  FROM TransactionHistory
  WHERE TransactionHistory.TransactionType = 'S'
  GROUP BY month, TransactionHistory.ProductID
),
ranked AS (
  SELECT
    monthly.*,
    ROW_NUMBER() OVER 

AgentAnswer(answer="The best-selling product by month is:\n- 2013-07: Classic Vest, S — 465 units\n- 2013-08: Water Bottle - 30 oz. — 513 units\n- 2013-09: Water Bottle - 30 oz. — 547 units\n- 2013-10: Water Bottle - 30 oz. — 555 units\n- 2013-11: Water Bottle - 30 oz. — 523 units\n- 2013-12: Water Bottle - 30 oz. — 476 units\n- 2014-01: Water Bottle - 30 oz. — 510 units\n\nI used sales transactions (TransactionType = 'S') and identified the top product by total quantity sold in each month.", sql_used="WITH monthly AS (\n  SELECT\n    strftime('%Y-%m', TransactionHistory.TransactionDate) AS month,\n    TransactionHistory.ProductID,\n    SUM(TransactionHistory.Quantity) AS total_qty\n  FROM TransactionHistory\n  WHERE TransactionHistory.TransactionType = 'S'\n  GROUP BY month, TransactionHistory.ProductID\n),\nranked AS (\n  SELECT\n    monthly.*, \n    ROW_NUMBER() OVER (PARTITION BY monthly.month ORDER BY monthly.total_qty DESC) AS rn\n  FROM monthly\n)\nSELECT ranked.month, ranked.Pr

## Logging: Langfuse

In [23]:
# Cell 2 — load env vars once at the top of notebook
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY      = os.getenv("OPENAI_API_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST       = os.getenv("LANGFUSE_HOST")

# verify all loaded
print("OPENAI_API_KEY:",      "✅" if OPENAI_API_KEY      else "❌ missing")
print("LANGFUSE_PUBLIC_KEY:", "✅" if LANGFUSE_PUBLIC_KEY else "❌ missing")
print("LANGFUSE_SECRET_KEY:", "✅" if LANGFUSE_SECRET_KEY else "❌ missing")

OPENAI_API_KEY: ✅
LANGFUSE_PUBLIC_KEY: ✅
LANGFUSE_SECRET_KEY: ✅


In [24]:
from langfuse import Langfuse

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST
)

# ── Test connection ───────────────────────────────────────────────
print("Langfuse connected:", langfuse.auth_check())

Langfuse connected: True


In [21]:
import time

def run_agent_with_langfuse(question: str) -> AgentAnswer:

    # ── Start trace — one per question ───────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    memory.add({"role": "user", "content": question})
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    logger.debug(f"messages in context: {len(messages)}")
    memory.stats()

    step = 0
    retries = 0
    last_df = None
    run_start = time.time()

    while True:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        memory.add(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            # ── Span — one per tool call ──────────────────────────
            span = trace.span(
                name=tc.function.name,
                input=args,
            )
            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"validation failed: {e}")
                        result = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        span.end(output=result, level="WARNING")
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached", level="ERROR")
                            return None
                        tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                        messages.append(tool_msg)
                        memory.add(tool_msg)
                        continue

                    logger.info(f"  sql: {validated.sql}")

                    try:
                        last_df = conn.execute(validated.sql).fetchdf()
                        result = last_df.to_string(index=False)
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    except Exception as e:
                        logger.error(f"sql failed: {e}")
                        result = f"SQL_ERROR: {str(e)}"
                        retries += 1
                        span.end(output=result, level="ERROR")
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached", level="ERROR")
                            return None
                        logger.warning(f"retry {retries}/{MAX_RETRIES}")
            finally:
                # ── Close span ────────────────────────────────────────
                span.end(output=result[:500])   # cap output size

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            memory.add(tool_msg)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from schema only. Set chart_type to none."
        )

        final = client.beta.chat.completions.parse(
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        trace.update(output=f"final answer error: {e}", level="ERROR")
        return None

    memory.add({"role": "assistant", "content": result.answer})

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart: {result.chart_type}")
    logger.info(f"duration: {duration}s")

    # ── Close trace ───────────────────────────────────────────────
    trace.update(
        output=result.answer,
        metadata={
            "chart_type": result.chart_type,
            "duration_seconds": duration,
            "steps": step,
            "sql_used": result.sql_used
        }
    )
    langfuse.flush()    # ensure trace is sent before notebook moves on

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [22]:
run_agent_with_langfuse('tell me about the routes of these products')

16:04:07 | INFO     | question: tell me about the routes of these products
  [memory] turns: 3 | summaries: 0 | tokens: 1556/6000
16:04:11 | INFO     | step 1: get_schema(WorkOrderRouting)
16:04:19 | INFO     | step 2: run_sql()
16:04:19 | INFO     |   sql: SELECT WorkOrderRouting.ProductID, Product.Name, WorkOrderRouting.OperationSequence, WorkOrderRouting.LocationID,
       COUNT(DISTINCT WorkOrderRouting.WorkOrderID) AS num_workorders,
       MIN(WorkOrderRouting.ScheduledStartDate) AS first_scheduled_start,
       MAX(WorkOrderRouting.ScheduledEndDate) AS last_scheduled_end,
       AVG(WorkOrderRouting.ActualResourceHrs) AS avg_resource_hrs,
       SUM(WorkOrderRouting.ActualResourceHrs) AS total_resource_hrs
FROM WorkOrderRouting
LEFT JOIN Product ON Product.ProductID = WorkOrderRouting.ProductID
WHERE WorkOrderRouting.ProductID IN (3, 870)
GROUP BY WorkOrderRouting.ProductID, Product.Name, WorkOrderRouting.OperationSequence, WorkOrderRouting.LocationID
ORDER BY WorkOrderRouting.P

AgentAnswer(answer='For the two products you asked about:\n- ProductID 3 (BB Ball Bearing) has 1,028 work orders, total ordered quantity of 822,830, total stocked quantity of 821,873, and total scrapped quantity of 957.\n- ProductID 870 (Water Bottle - 30 oz.) has no work orders in this dataset.\n- Neither product has any recorded routing rows in WorkOrderRouting (routing_rows = 0 for both), so there are no operation sequences or locations available for these products here.\n\nIn short: BB Ball Bearing is manufactured in this dataset but has no detailed routing records, while Water Bottle - 30 oz. appears to have neither manufacturing work orders nor routing records.', sql_used='SELECT p.ProductID, p.Name, p.MakeFlag, p.FinishedGoodsFlag, p.ProductLine, p.Class, p.Style, p.ListPrice, p.StandardCost,\n       COUNT(DISTINCT wr.WorkOrderID) AS routing_rows,\n       COUNT(DISTINCT wo.WorkOrderID) AS num_workorders,\n       SUM(wo.OrderQty) AS total_order_qty,\n       SUM(wo.StockedQty) AS 

# Failure Handling & Rate Limits

In [25]:
import time
import random
from openai import RateLimitError, APITimeoutError, APIConnectionError, APIStatusError

def call_openai(fn, *args, **kwargs):
    """
    Wraps any OpenAI API call with exponential backoff.
    Usage: call_openai(client.chat.completions.create, model=..., messages=...)
    """
    max_retries = 4
    base_wait   = 1     # seconds

    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)

        except RateLimitError as e:
            wait = base_wait * (2 ** attempt) + random.uniform(0, 1)
            logger.warning(f"rate limit hit — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APITimeoutError as e:
            wait = base_wait * (2 ** attempt)
            logger.warning(f"timeout — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APIConnectionError as e:
            wait = base_wait * (2 ** attempt)
            logger.warning(f"connection error — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APIStatusError as e:
            # 500s are server errors — retry
            # 400s are client errors — don't retry
            if e.status_code >= 500:
                wait = base_wait * (2 ** attempt)
                logger.warning(f"server error {e.status_code} — waiting {wait:.1f}s")
                time.sleep(wait)
            else:
                logger.error(f"client error {e.status_code}: {e.message}")
                raise   # don't retry 400s — it won't help

        except Exception as e:
            logger.error(f"unexpected error: {type(e).__name__}: {e}")
            raise

    logger.error(f"all {max_retries} attempts failed")
    return None

In [29]:
# ── Test ──────────────────────────────────────────────────────────
response = call_openai(
    client.chat.completions.create,
    model=AGENT_MODEL,
    messages=[{"role": "user", "content": "say hello"}],
    tools=TOOLS
)
print(response.choices[0].message.content)

Hello! How can I help you today?


In [40]:
import time

def run_agent_with_failure_handling(question: str) -> AgentAnswer:

    # ── Start trace — one per question ───────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    memory.add_question(question)
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    logger.debug(f"messages in context: {len(messages)}")
    memory.stats()

    step = 0
    retries = 0
    last_df = None
    run_start = time.time()

    while True:
        response = call_openai(
            client.chat.completions.create,
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        # memory.add(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            # ── Span — one per tool call ──────────────────────────
            span = trace.span(
                name=tc.function.name,
                input=args,
            )
            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"validation failed: {e}")
                        result = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        span.end(output=result, level="WARNING")
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached", level="ERROR")
                            return None
                        tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                        messages.append(tool_msg)
                        # memory.add(tool_msg)
                        continue

                    logger.info(f"  sql: {validated.sql}")

                    try:
                        last_df = conn.execute(validated.sql).fetchdf()
                        result = last_df.to_string(index=False)
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    except Exception as e:
                        logger.error(f"sql failed: {e}")
                        result = f"SQL_ERROR: {str(e)}"
                        retries += 1
                        span.end(output=result, level="ERROR")
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached", level="ERROR")
                            return None
                        logger.warning(f"retry {retries}/{MAX_RETRIES}")
            finally:
                # ── Close span ────────────────────────────────────────
                span.end(output=result[:500])   # cap output size

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            # memory.add(tool_msg)


    # Recheck tokens
    messages = memory.get_messages(SYSTEM_PROMPT)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from schema only. Set chart_type to none."
        )

        final = call_openai(
            client.beta.chat.completions.parse,
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )

        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        trace.update(output=f"final answer error: {e}", level="ERROR")
        return None

    # memory.add({"role": "assistant", "content": result.answer})
    memory.add_answer(result.answer)

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart: {result.chart_type}")
    logger.info(f"duration: {duration}s")

    # ── Close trace ───────────────────────────────────────────────
    trace.update(
        output=result.answer,
        metadata={
            "chart_type": result.chart_type,
            "duration_seconds": duration,
            "steps": step,
            "sql_used": result.sql_used
        }
    )
    langfuse.flush()    # ensure trace is sent before notebook moves on

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [61]:
run_agent_with_failure_handling('hlp me with workorder of this product')


09:12:59 | INFO     | question: hlp me with workorder of this product
  [memory] turns: 3/2 | cached tables: 2 | summary: no | tokens: unknown
09:13:05 | INFO     | step 1: list_tables()
09:13:05 | INFO     |   list_tables: from cache
  [memory] summarised — keeping last 2 turns
09:13:21 | INFO     | answer: From schema only: to analyze the work orders for a product, the relevant tables are Product, WorkOrd...
09:13:21 | INFO     | chart: none
09:13:21 | INFO     | duration: 21.43s


AgentAnswer(answer='From schema only: to analyze the work orders for a product, the relevant tables are Product, WorkOrder, and possibly WorkOrderRouting. Product identifies the product. WorkOrder should contain the manufacturing work orders tied to that product. WorkOrderRouting would likely provide the step-by-step routing or operation details for each work order. I cannot identify the specific work orders for your product without running SQL.', sql_used='', chart_type='none', plotly_code='')

## memory correction

In [38]:
import tiktoken

class MemoryManager:

    def __init__(self, model: str, max_turns: int = 6, max_tokens: int = 6000):
        self.model      = model
        self.max_turns  = max_turns
        self.max_tokens = max_tokens
        self.encoder    = tiktoken.encoding_for_model(model)
        self.history     = []    # Q&A pairs only
        self.summary     = None  # rolling summary of older turns
        self.tables_cache = None # list_tables result
        self.schema_cache = {}   # {table_name: schema string}

    # ── Token counting ────────────────────────────────────────────
    def count_tokens(self, messages: list) -> int:
        return sum(
            len(self.encoder.encode(m["content"]))
            for m in messages
            if isinstance(m.get("content"), str)
        )

    # ── Cache ─────────────────────────────────────────────────────
    def cache_tables(self, result: str):
        self.tables_cache = result
        logger.info(f"  [memory] tables cached")

    def cache_schema(self, table_name: str, result: str):
        self.schema_cache[table_name] = result
        logger.info(f"  [memory] schema cached: {table_name}")

    def get_cached_tables(self) -> str | None:
        return self.tables_cache

    def get_cached_schema(self, table_name: str) -> str | None:
        return self.schema_cache.get(table_name)

    # ── Q&A ───────────────────────────────────────────────────────
    def add_question(self, question: str):
        self.history.append({"role": "user", "content": question})

    def add_answer(self, answer: str):
        self.history.append({"role": "assistant", "content": answer})
        self._maybe_summarise()

    def _maybe_summarise(self):
        """Triggered after every answer. Acts only when turns exceed max."""
        turns = len([m for m in self.history if m["role"] == "user"])
        if turns <= self.max_turns:
            return

        # split — keep last max_turns, summarise the rest
        keep_from = -(self.max_turns * 2)
        to_summarise = self.history[:keep_from]
        self.history  = self.history[keep_from:]

        prior = f"Previous summary:\n{self.summary}\n\nNew turns:\n" if self.summary else ""
        text  = prior + "\n".join(
            f"{m['role']}: {m['content']}"
            for m in to_summarise
            if isinstance(m.get("content"), str)
        )

        tokens_before = self.count_tokens(self.get_messages(""))
        logger.info(f"  [memory] summarising — tokens before: {tokens_before}")

        response = client.chat.completions.create(
            model=self.model,
            messages=[{
                "role": "user",
                "content": f"Summarise this conversation. Keep key questions, findings, numbers, insights. 4-5 sentences.\n\n{text}"
            }]
        )
        self.summary = response.choices[0].message.content

        tokens_after = self.count_tokens(self.get_messages(""))
        logger.info(f"  [memory] summarised — tokens after: {tokens_after}")

    # ── Build messages ────────────────────────────────────────────
    def get_messages(self, system_prompt: str) -> list:
        messages = [{"role": "system", "content": system_prompt}]

        if self.tables_cache:
            messages.append({
                "role": "system",
                "content": f"[Available tables]: {self.tables_cache}"
            })

        if self.schema_cache:
            schema_text = "\n\n".join(
                f"[{t}]:\n{s}"
                for t, s in self.schema_cache.items()
            )
            messages.append({
                "role": "system",
                "content": f"[Table schemas]:\n{schema_text}"
            })

        if self.summary:
            messages.append({
                "role": "system",
                "content": f"[Earlier context]: {self.summary}"
            })

        messages.extend(self.history)
        return messages

    # ── Stats ─────────────────────────────────────────────────────
    def stats(self, system_prompt: str = ""):
        tokens = self.count_tokens(self.get_messages(system_prompt))
        turns  = len([m for m in self.history if m["role"] == "user"])
        print(f"  [memory] turns: {turns} | cached schemas: {len(self.schema_cache)} | summary: {'yes' if self.summary else 'no'} | tokens: {tokens}/{self.max_tokens}")

    def reset(self):
        self.history  = []
        self.summary  = None
        # cache preserved intentionally
        print("  [memory] reset — cache preserved")

In [44]:
memory = MemoryManager(model=AGENT_MODEL, max_turns=2, max_tokens=6000)


In [37]:
SYSTEM_PROMPT = """You are a data analyst agent with access to a DuckDB database.

CONTEXT RULES:
- If [Available tables] is in context → do NOT call list_tables
- If [Table schemas] contains the table you need → do NOT call get_schema
- Always run SQL to answer data questions — never guess from context alone

PROCESS:
1. Check context for available schema
2. Fetch only missing schemas
3. Always run_sql for any data or comparison question
4. Return final structured answer

SQL rules:
- No LIMIT on GROUP BY queries
- LIMIT 100 on raw rows
- Always qualify column names with table name
- Prefer LEFT JOIN unless INNER JOIN clearly needed
- Never assume dates or years — always query what exists first

plotly_code rules:
- df is already loaded as a pandas DataFrame
- create figure called fig
- do NOT call fig.show()
- use double quotes only — never single quotes
"""

In [48]:
def execute_tool(tool_name: str, tool_args: dict) -> tuple:
    """Returns (result_string, dataframe_or_none)"""

    if tool_name == "list_tables":
        cached = memory.get_cached_tables()
        if cached:
            logger.info("  list_tables: from cache")
            return cached, None
        tables = conn.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
        """).fetchdf()["table_name"].tolist()
        result = str(tables)
        memory.cache_tables(result)
        return result, None

    elif tool_name == "get_schema":
        table = tool_args.get("table_name")
        cached = memory.get_cached_schema(table)
        if cached:
            logger.info(f"  get_schema({table}): from cache")
            return cached, None
        cols   = conn.execute(f"DESCRIBE {table}").fetchdf()
        count  = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        sample = conn.execute(f"SELECT * FROM {table} LIMIT 1").fetchdf()
        result = (
            f"TABLE: {table} | rows: {count}\n"
            f"COLUMNS:\n{cols[['column_name','column_type']].to_string(index=False)}\n"
            f"SAMPLE:\n{sample.to_string(index=False)}"
        )
        memory.cache_schema(table, result)
        return result, None

    elif tool_name == "run_sql":
        try:
            validated = RunSqlInput(**tool_args)
            df = conn.execute(validated.sql).fetchdf()
            return df.to_string(index=False), df
        except Exception as e:
            return f"SQL_ERROR: {str(e)}", None

    return f"UNKNOWN_TOOL: {tool_name}", None

In [77]:
import time

MAX_RETRIES  = 3
AGENT_MODEL  = "gpt-4.1-mini"
FINAL_MODEL  = "gpt-5-mini"

memory = MemoryManager(model=AGENT_MODEL, max_turns=2, max_tokens=6000)


def run_agent(question: str) -> AgentAnswer:

    # ── Trace ─────────────────────────────────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    # ── Memory ────────────────────────────────────────────────────
    memory.add_question(question)
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    memory.stats(SYSTEM_PROMPT)

    step     = 0
    retries  = 0
    last_df  = None
    run_start = time.time()

    while True:
        response = call_openai(
            client.chat.completions.create,
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )
        if response is None:
            logger.error("agent loop: all retries failed")
            trace.update(output="openai call failed")
            return None

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)
            span = trace.span(name=tc.function.name, input=args)

            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result, _ = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result, _ = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"  validation failed: {e}")
                        result   = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                        continue

                    logger.info(f"  sql: {validated.sql}")
                    result, last_df = execute_tool("run_sql", {"sql": validated.sql})

                    if last_df is not None:
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    else:
                        logger.error(f"  sql failed: {result}")
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        logger.warning(f"  retry {retries}/{MAX_RETRIES}")

            finally:
                span.end(output=str(result)[:500])

            # tool result stays in messages this turn only — never in memory
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    # ── Final answer ──────────────────────────────────────────────
    try:
        df_columns = [str(col) for col in last_df.columns] if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from context only. Set chart_type to none."
        )

        final = call_openai(
            client.beta.chat.completions.parse,
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        if final is None:
            logger.error("final answer: all retries failed")
            trace.update(output="final answer call failed")
            return None

        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        # fallback — no chart
        try:
            fallback = call_openai(
                client.beta.chat.completions.parse,
                model=FINAL_MODEL,
                messages=messages + [{
                    "role": "user",
                    "content": f"{df_context} Give final answer. Set plotly_code to empty string and chart_type to none."
                }],
                response_format=AgentAnswer
            )
            if fallback is None:
                return None
            result = fallback.choices[0].message.parsed
            logger.warning("fallback answer used — no chart")
        except Exception as e2:
            logger.error(f"fallback failed: {e2}")
            trace.update(output=f"final answer error: {e2}")
            return None

    # ── Memory — Q&A only ─────────────────────────────────────────
    memory.add_answer(result.answer)

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart:  {result.chart_type}")
    logger.info(f"duration: {duration}s")

    trace.update(
        output=result.answer,
        metadata={
            "chart_type":       result.chart_type,
            "duration_seconds": duration,
            "steps":            step,
            "sql_used":         result.sql_used
        }
    )
    langfuse.flush()

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [78]:
run_agent("which product has the most transactions?")

10:58:50 | INFO     | question: which product has the most transactions?
  [memory] turns: 1 | cached schemas: 0 | summary: no | tokens: 188/6000
10:58:51 | INFO     | step 1: list_tables()
10:58:51 | INFO     |   [memory] tables cached
10:58:53 | INFO     | step 2: get_schema(Product)
10:58:53 | INFO     |   [memory] schema cached: Product
10:58:53 | INFO     | step 3: get_schema(TransactionHistory)
10:58:53 | INFO     |   [memory] schema cached: TransactionHistory
10:58:54 | INFO     | step 4: run_sql()
10:58:54 | INFO     |   sql: SELECT Product.ProductID, Product.Name, COUNT(TransactionHistory.TransactionID) AS TransactionCount
FROM TransactionHistory
LEFT JOIN Product ON TransactionHistory.ProductID = Product.ProductID
GROUP BY Product.ProductID, Product.Name
ORDER BY TransactionCount DESC
LIMIT 1;
10:58:54 | INFO     |   rows returned: 1
10:59:05 | INFO     | answer: The product with the most transactions is "Water Bottle - 30 oz." (ProductID 870) with 2339 transact...
10:59:05 |

AgentAnswer(answer='The product with the most transactions is "Water Bottle - 30 oz." (ProductID 870) with 2339 transactions.', sql_used='SELECT Product.ProductID, Product.Name, COUNT(TransactionHistory.TransactionID) AS TransactionCount\nFROM TransactionHistory\nLEFT JOIN Product ON TransactionHistory.ProductID = Product.ProductID\nGROUP BY Product.ProductID, Product.Name\nORDER BY TransactionCount DESC\nLIMIT 1;', chart_type='bar', plotly_code='import plotly.express as px\n\n# df is assumed to contain columns: [\'ProductID\', \'Name\', \'TransactionCount\']\nfig = px.bar(df, x="Name", y="TransactionCount", hover_data=["ProductID"],\n             title="Transactions per Product (Top Product)",\n             labels={"Name":"Product Name", "TransactionCount":"Transaction Count"})\nfig.update_layout(xaxis_tickangle=-45)')

In [79]:
run_agent("this products month on month transactions")

10:59:09 | INFO     | question: this products month on month transactions
  [memory] turns: 2 | cached schemas: 2 | summary: no | tokens: 771/6000
10:59:10 | INFO     | step 1: run_sql()
10:59:10 | INFO     |   sql: SELECT
  EXTRACT(YEAR FROM TransactionHistory.TransactionDate) AS Year,
  EXTRACT(MONTH FROM TransactionHistory.TransactionDate) AS Month,
  COUNT(TransactionHistory.TransactionID) AS TransactionCount
FROM TransactionHistory
WHERE TransactionHistory.ProductID = 870
GROUP BY Year, Month
ORDER BY Year, Month;
10:59:10 | INFO     |   rows returned: 7
10:59:24 | INFO     | answer: Month-on-month transaction counts for ProductID 870 (Water Bottle - 30 oz.):
July 2013: 54
August 20...
10:59:24 | INFO     | chart:  line
10:59:24 | INFO     | duration: 15.61s


AgentAnswer(answer='Month-on-month transaction counts for ProductID 870 (Water Bottle - 30 oz.):\nJuly 2013: 54\nAugust 2013: 348\nSeptember 2013: 350\nOctober 2013: 359\nNovember 2013: 421\nDecember 2013: 382\nJanuary 2014: 425', sql_used='SELECT\n  EXTRACT(YEAR FROM TransactionHistory.TransactionDate) AS Year,\n  EXTRACT(MONTH FROM TransactionHistory.TransactionDate) AS Month,\n  COUNT(TransactionHistory.TransactionID) AS TransactionCount\nFROM TransactionHistory\nWHERE TransactionHistory.ProductID = 870\nGROUP BY Year, Month\nORDER BY Year, Month;', chart_type='line', plotly_code='import pandas as pd\nimport plotly.express as px\n\n# Create a datetime x-axis from Year and Month\ndf_dates = df.copy()\ndf_dates["Date"] = pd.to_datetime(df_dates["Year"].astype(int).astype(str) + "-" + df_dates["Month"].astype(int).astype(str) + "-01")\n\nfig = px.line(df_dates, x="Date", y="TransactionCount", markers=True, title="Monthly Transactions for ProductID 870")\nfig.update_xaxes(dtick="M1", ti

In [80]:
run_agent("lets compare this product sales vs others in nov month")

10:59:29 | INFO     | question: lets compare this product sales vs others in nov month
  [memory] turns: 3 | cached schemas: 2 | summary: no | tokens: 855/6000
10:59:30 | INFO     | step 1: run_sql()
10:59:30 | INFO     |   sql: SELECT
  ProductID,
  SUM(Quantity) AS TotalQuantity
FROM TransactionHistory
WHERE TransactionType = 'S'
  AND EXTRACT(MONTH FROM TransactionDate) = 11
GROUP BY ProductID
ORDER BY TotalQuantity DESC;
10:59:30 | INFO     |   rows returned: 145
10:59:31 | INFO     | step 2: run_sql()
10:59:31 | INFO     |   sql: SELECT ProductID, Name FROM Product WHERE ProductID = 870;
10:59:31 | INFO     |   rows returned: 1
10:59:33 | INFO     | step 3: run_sql()
10:59:33 | INFO     |   sql: SELECT ProductID, Name FROM Product WHERE ProductID IN (712, 707, 873, 711);
10:59:33 | INFO     |   rows returned: 4
11:00:00 | INFO     |   [memory] summarising — tokens before: 671
11:00:01 | INFO     |   [memory] summarised — tokens after: 737
11:00:01 | INFO     | answer: I will provi

AgentAnswer(answer="I will provide the November sales comparison (top products by quantity) and a Plotly bar chart using only the df columns ['ProductID', 'Name'].", sql_used="SELECT th.ProductID, p.Name, SUM(th.Quantity) AS TotalQuantity\nFROM TransactionHistory th\nLEFT JOIN Product p ON th.ProductID = p.ProductID\nWHERE th.TransactionType = 'S'\n  AND EXTRACT(MONTH FROM th.TransactionDate) = 11\nGROUP BY th.ProductID, p.Name\nORDER BY TotalQuantity DESC;", chart_type='bar', plotly_code='# df contains columns [\'ProductID\', \'Name\'] for the top products in November\n# quantities list corresponds to those rows in the same order\nquantities = [523, 319, 296, 284, 281, 273, 267, 217, 203, 202]\nimport plotly.graph_objects as go\nfig = go.Figure()\nfig.add_trace(go.Bar(x=quantities, y=df["Name"].head(10), orientation="h", marker_color="steelblue", text=quantities, textposition="auto"))\nfig.update_layout(title_text="Top 10 Products by Quantity Sold in November", xaxis_title="Quantity S

## evaluation

In [82]:
from pydantic import BaseModel, Field

class EvaluationResult(BaseModel):
    answer_relevance:       float = Field(..., ge=0, le=1, description="Did the answer address the question?")
    sql_correctness:        float = Field(..., ge=0, le=1, description="Did the SQL match the question intent?")
    chart_appropriateness:  float = Field(..., ge=0, le=1, description="Was the chart type appropriate for the data?")
    tool_efficiency:        float = Field(..., ge=0, le=1, description="Were minimum tool calls used?")
    reasoning:              str   = Field(..., description="Brief explanation of scores")


def evaluate_run(
    question:   str,
    answer:     str,
    sql_used:   str,
    chart_type: str,
    steps:      int
) -> EvaluationResult:
    """
    LLM-as-a-judge. Uses cheap model — evaluation does not need gpt-4o.
    Returns structured scores for each dimension.
    """

    prompt = f"""You are evaluating a data analyst AI agent. Score each dimension from 0 to 1.

QUESTION ASKED:
{question}

AGENT ANSWER:
{answer}

SQL USED:
{sql_used}

CHART TYPE CHOSEN:
{chart_type}

TOOL CALLS MADE:
{steps} steps total

SCORING GUIDE:
answer_relevance:
  1.0 = directly and completely answers the question
  0.5 = partially answers, missing key details
  0.0 = irrelevant or wrong answer

sql_correctness:
  1.0 = SQL clearly matches the question intent
  0.5 = SQL runs but may miss edge cases
  0.0 = SQL is wrong or missing entirely

chart_appropriateness:
  1.0 = perfect chart type for this data
  0.5 = acceptable but not ideal
  0.0 = wrong chart type or none when needed

tool_efficiency:
  1.0 = minimum tools used, no redundant calls
  0.5 = some redundant calls but acceptable
  0.0 = excessive redundant tool calls

Score honestly. Be strict."""

    response = call_openai(
        client.beta.chat.completions.parse,
        model=AGENT_MODEL,    # cheap model for evaluation
        messages=[{"role": "user", "content": prompt}],
        response_format=EvaluationResult
    )

    return response.choices[0].message.parsed

In [83]:
# ── Test ──────────────────────────────────────────────────────────
test_eval = evaluate_run(
    question   = "which product has the most transactions?",
    answer     = "Water Bottle - 30 oz. has the most transactions with 3688.",
    sql_used   = "SELECT p.Name, COUNT(*) FROM TransactionHistory t JOIN Product p ON t.ProductID = p.ProductID GROUP BY p.Name ORDER BY COUNT(*) DESC LIMIT 1",
    chart_type = "bar",
    steps      = 3
)

print(f"answer_relevance:      {test_eval.answer_relevance}")
print(f"sql_correctness:       {test_eval.sql_correctness}")
print(f"chart_appropriateness: {test_eval.chart_appropriateness}")
print(f"tool_efficiency:       {test_eval.tool_efficiency}")
print(f"reasoning:             {test_eval.reasoning}")

answer_relevance:      1.0
sql_correctness:       1.0
chart_appropriateness: 0.5
tool_efficiency:       1.0
reasoning:             The answer directly and correctly identifies the product with the most transactions. The SQL statement perfectly matches the question by counting transactions per product and selecting the top one. While a bar chart can display this data, since the question asks for the single top product, a bar chart is not the most ideal visualization (a single value or simple highlight may suffice), but it is acceptable. The agent used the minimum number of steps/tool calls needed without redundancies.


In [85]:
import time

MAX_RETRIES  = 3
AGENT_MODEL  = "gpt-4.1-mini"
FINAL_MODEL  = "gpt-5-mini"

memory = MemoryManager(model=AGENT_MODEL, max_turns=2, max_tokens=6000)


def run_agent_with_evaluation(question: str) -> AgentAnswer:

    # ── Trace ─────────────────────────────────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    # ── Memory ────────────────────────────────────────────────────
    memory.add_question(question)
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    memory.stats(SYSTEM_PROMPT)

    step     = 0
    retries  = 0
    last_df  = None
    run_start = time.time()

    while True:
        response = call_openai(
            client.chat.completions.create,
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )
        if response is None:
            logger.error("agent loop: all retries failed")
            trace.update(output="openai call failed")
            return None

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)
            span = trace.span(name=tc.function.name, input=args)

            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result, _ = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result, _ = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"  validation failed: {e}")
                        result   = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                        continue

                    logger.info(f"  sql: {validated.sql}")
                    result, last_df = execute_tool("run_sql", {"sql": validated.sql})

                    if last_df is not None:
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    else:
                        logger.error(f"  sql failed: {result}")
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        logger.warning(f"  retry {retries}/{MAX_RETRIES}")

            finally:
                span.end(output=str(result)[:500])

            # tool result stays in messages this turn only — never in memory
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    # ── Final answer ──────────────────────────────────────────────
    try:
        df_columns = [str(col) for col in last_df.columns] if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from context only. Set chart_type to none."
        )

        final = call_openai(
            client.beta.chat.completions.parse,
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        if final is None:
            logger.error("final answer: all retries failed")
            trace.update(output="final answer call failed")
            return None

        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        # fallback — no chart
        try:
            fallback = call_openai(
                client.beta.chat.completions.parse,
                model=FINAL_MODEL,
                messages=messages + [{
                    "role": "user",
                    "content": f"{df_context} Give final answer. Set plotly_code to empty string and chart_type to none."
                }],
                response_format=AgentAnswer
            )
            if fallback is None:
                return None
            result = fallback.choices[0].message.parsed
            logger.warning("fallback answer used — no chart")
        except Exception as e2:
            logger.error(f"fallback failed: {e2}")
            trace.update(output=f"final answer error: {e2}")
            return None

    # ── Memory — Q&A only ─────────────────────────────────────────
    memory.add_answer(result.answer)

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart:  {result.chart_type}")
    logger.info(f"duration: {duration}s")

    trace.update(
        output=result.answer,
        metadata={
            "chart_type":       result.chart_type,
            "duration_seconds": duration,
            "steps":            step,
            "sql_used":         result.sql_used
        }
    )
    langfuse.flush()

    # ── Evaluation ────────────────────────────────────────────────
    try:
        eval_result = evaluate_run(
            question   = question,
            answer     = result.answer,
            sql_used   = result.sql_used,
            chart_type = result.chart_type,
            steps      = step
        )

        # send scores to langfuse trace
        trace.score(name="answer_relevance",      value=eval_result.answer_relevance)
        trace.score(name="sql_correctness",       value=eval_result.sql_correctness)
        trace.score(name="chart_appropriateness", value=eval_result.chart_appropriateness)
        trace.score(name="tool_efficiency",       value=eval_result.tool_efficiency)

        langfuse.flush()

        logger.info(f"eval — relevance: {eval_result.answer_relevance} | sql: {eval_result.sql_correctness} | chart: {eval_result.chart_appropriateness} | efficiency: {eval_result.tool_efficiency}")
        logger.debug(f"eval reasoning: {eval_result.reasoning}")

    except Exception as e:
        logger.warning(f"evaluation failed: {e}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [86]:
run_agent_with_evaluation('top 3 transacted prodcuts in dec')

17:36:56 | INFO     | question: top 3 transacted prodcuts in dec
  [memory] turns: 1 | cached schemas: 0 | summary: no | tokens: 190/6000
17:36:58 | INFO     | step 1: list_tables()
17:36:58 | INFO     |   [memory] tables cached
17:36:59 | INFO     | step 2: get_schema(TransactionHistory)
17:36:59 | INFO     |   [memory] schema cached: TransactionHistory
17:37:00 | INFO     | step 3: run_sql()
17:37:00 | INFO     |   sql: SELECT ProductID, SUM(Quantity) AS TotalQuantity
FROM TransactionHistory
WHERE EXTRACT(MONTH FROM TransactionDate) = 12
GROUP BY ProductID
ORDER BY TotalQuantity DESC
LIMIT 3;
17:37:00 | INFO     |   rows returned: 3
17:37:02 | INFO     | step 4: get_schema(Product)
17:37:02 | INFO     |   [memory] schema cached: Product
17:37:03 | INFO     | step 5: run_sql()
17:37:03 | INFO     |   sql: SELECT Product.ProductID, Product.Name, TH.TotalQuantity
FROM Product
JOIN (
    SELECT ProductID, SUM(Quantity) AS TotalQuantity
    FROM TransactionHistory
    WHERE EXTRACT(MONTH 

AgentAnswer(answer='Top 3 transacted products in December:\n1. BB Ball Bearing — 24,950\n2. Seat Stays — 11,444\n3. HL Crankarm — 6,600', sql_used='SELECT Product.ProductID, Product.Name, TH.TotalQuantity\nFROM Product\nJOIN (\n    SELECT ProductID, SUM(Quantity) AS TotalQuantity\n    FROM TransactionHistory\n    WHERE EXTRACT(MONTH FROM TransactionDate) = 12\n    GROUP BY ProductID\n    ORDER BY TotalQuantity DESC\n    LIMIT 3\n) AS TH ON Product.ProductID = TH.ProductID\nORDER BY TH.TotalQuantity DESC;', chart_type='bar', plotly_code='import plotly.express as px\n\nfig = px.bar(df, x="Name", y="TotalQuantity", text="TotalQuantity")\nfig.update_layout(title="Top 3 Transacted Products in December", xaxis_title="Product Name", yaxis_title="Total Quantity")\nfig.update_traces(marker_color="steelblue", texttemplate="%{text:.0f}", textposition="outside")')

## cost optimisation

In [109]:
class CostTracker:

    def __init__(self):
        self.input_tokens  = 0
        self.output_tokens = 0

    def add(self, input_tokens: int, output_tokens: int):
        self.input_tokens  += input_tokens
        self.output_tokens += output_tokens

    def log(self):
        logger.info(f"cost — input: {self.input_tokens} | output: {self.output_tokens} tokens")

    def reset(self):
        self.input_tokens  = 0
        self.output_tokens = 0

In [111]:
# ── Constants — top of notebook ───────────────────────────────────
MODEL_COSTS = {
    "gpt-4.1-mini": {"input": 0.4,  "output": 1.60},
    "gpt-5-mini":  {"input": 0.25,  "output": 2.00},
    "gpt-5.4":     {"input": 2.50,  "output": 15.00},
}

# ── Initialise once ───────────────────────────────────────────────
cost_tracker = CostTracker()

In [102]:
# ── Test ──────────────────────────────────────────────────────────
cost_tracker.track("gpt-4o-mini", input_tokens=500, output_tokens=100)
cost_tracker.track("gpt-4o",      input_tokens=200, output_tokens=50)
cost_tracker.log()
print(cost_tracker.summary())
cost_tracker.reset()

18:04:42 | INFO     | cost — input: 700 | output: 150 | total: $0.000000
{'input_tokens': 700, 'output_tokens': 150, 'total_cost_usd': 0.0}


In [112]:
def call_openai(fn, *args, **kwargs):
    max_retries = 4
    base_wait   = 1

    for attempt in range(max_retries):
        try:
            response = fn(*args, **kwargs)
            return response

        except RateLimitError as e:
            wait = base_wait * (2 ** attempt) + random.uniform(0, 1)
            logger.warning(f"rate limit — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APITimeoutError:
            wait = base_wait * (2 ** attempt)
            logger.warning(f"timeout — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APIConnectionError:
            wait = base_wait * (2 ** attempt)
            logger.warning(f"connection error — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APIStatusError as e:
            if e.status_code >= 500:
                wait = base_wait * (2 ** attempt)
                logger.warning(f"server error {e.status_code} — waiting {wait:.1f}s")
                time.sleep(wait)
            else:
                logger.error(f"client error {e.status_code}: {e.message}")
                raise

        except Exception as e:
            logger.error(f"unexpected error: {type(e).__name__}: {e}")
            raise

    logger.error(f"all {max_retries} attempts failed")
    return None

In [113]:
import time

MAX_RETRIES = 3
AGENT_MODEL = "gpt-4.1-mini"
FINAL_MODEL = "gpt-5-mini"

memory = MemoryManager(model=AGENT_MODEL, max_turns=2, max_tokens=6000)


def run_agent_with_cost(question: str) -> AgentAnswer:

    # ── Trace ─────────────────────────────────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    # ── Memory ────────────────────────────────────────────────────
    memory.add_question(question)
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    memory.stats(SYSTEM_PROMPT)

    step      = 0
    retries   = 0
    last_df   = None
    run_start = time.time()

    while True:

        # ── LLM call — generation for cost tracking ───────────────
        gen = trace.generation(
            name  = f"agent-loop-{step}",
            model = AGENT_MODEL,
            input = messages
        )

        response = call_openai(
            client.chat.completions.create,
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        if response is None:
            logger.error("agent loop: all retries failed")
            gen.end(output="failed")
            trace.update(output="openai call failed")
            return None

        msg = response.choices[0].message
        messages.append(msg)

        gen.end(
            output=msg.content or "",
            usage={
                "input":  response.usage.prompt_tokens,
                "output": response.usage.completion_tokens
            }
        )

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            # ── Tool call — span ──────────────────────────────────
            span = trace.span(name=tc.function.name, input=args)

            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result, _ = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result, _ = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"  validation failed: {e}")
                        result   = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                        continue

                    logger.info(f"  sql: {validated.sql}")
                    result, last_df = execute_tool("run_sql", {"sql": validated.sql})

                    if last_df is not None:
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    else:
                        logger.error(f"  sql failed: {result}")
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        logger.warning(f"  retry {retries}/{MAX_RETRIES}")

            finally:
                span.end(output=str(result)[:500])

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    # ── Final answer ──────────────────────────────────────────────
    try:
        df_columns = [str(col) for col in last_df.columns] if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from context only. Set chart_type to none."
        )

        gen_final = trace.generation(
            name  = "final-answer",
            model = FINAL_MODEL,
            input = messages
        )

        final = call_openai(
            client.beta.chat.completions.parse,
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )

        if final is None:
            logger.error("final answer: all retries failed")
            gen_final.end(output="failed")
            trace.update(output="final answer call failed")
            return None

        result = final.choices[0].message.parsed

        gen_final.end(
            output=result.answer,
            usage={
                "input":  final.usage.prompt_tokens,
                "output": final.usage.completion_tokens
            }
        )

    except Exception as e:
        logger.error(f"final answer error: {e}")
        try:
            fallback = call_openai(
                client.beta.chat.completions.parse,
                model=FINAL_MODEL,
                messages=messages + [{
                    "role": "user",
                    "content": f"{df_context} Give final answer. Set plotly_code to empty string and chart_type to none."
                }],
                response_format=AgentAnswer
            )
            if fallback is None:
                return None
            result = fallback.choices[0].message.parsed
            logger.warning("fallback answer used — no chart")
        except Exception as e2:
            logger.error(f"fallback failed: {e2}")
            trace.update(output=f"final answer error: {e2}")
            return None

    # ── Memory ────────────────────────────────────────────────────
    memory.add_answer(result.answer)

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart:  {result.chart_type}")
    logger.info(f"duration: {duration}s")

    # ── Trace update — no token fields, langfuse has them ─────────
    trace.update(
        output=result.answer,
        metadata={
            "chart_type":       result.chart_type,
            "duration_seconds": duration,
            "steps":            step,
            "sql_used":         result.sql_used
        }
    )
    langfuse.flush()

    # ── Evaluation ────────────────────────────────────────────────
    try:
        eval_result = evaluate_run(
            question   = question,
            answer     = result.answer,
            sql_used   = result.sql_used,
            chart_type = result.chart_type,
            steps      = step
        )

        trace.score(name="answer_relevance",      value=eval_result.answer_relevance)
        trace.score(name="sql_correctness",       value=eval_result.sql_correctness)
        trace.score(name="chart_appropriateness", value=eval_result.chart_appropriateness)
        trace.score(name="tool_efficiency",       value=eval_result.tool_efficiency)

        langfuse.flush()

        logger.info(f"eval — relevance: {eval_result.answer_relevance} | sql: {eval_result.sql_correctness} | chart: {eval_result.chart_appropriateness} | efficiency: {eval_result.tool_efficiency}")
        logger.debug(f"eval reasoning: {eval_result.reasoning}")

    except Exception as e:
        logger.warning(f"evaluation failed: {e}")

    # ── Chart ─────────────────────────────────────────────────────
    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [114]:
run_agent_with_cost("which few product has the most transactions?")


18:34:00 | INFO     | question: which few product has the most transactions?
  [memory] turns: 1 | cached schemas: 0 | summary: no | tokens: 189/6000
18:34:01 | INFO     | step 1: list_tables()
18:34:01 | INFO     |   [memory] tables cached
18:34:03 | INFO     | step 2: get_schema(Product)
18:34:03 | INFO     |   [memory] schema cached: Product
18:34:03 | INFO     | step 3: get_schema(TransactionHistory)
18:34:03 | INFO     |   [memory] schema cached: TransactionHistory
18:34:04 | INFO     | step 4: run_sql()
18:34:04 | INFO     |   sql: SELECT p.Name, COUNT(th.TransactionID) AS TransactionCount
FROM TransactionHistory th
JOIN Product p ON th.ProductID = p.ProductID
GROUP BY p.Name
ORDER BY TransactionCount DESC
LIMIT 5;
18:34:04 | INFO     |   rows returned: 5
18:34:20 | INFO     | answer: Top 5 products by transaction count:
1. Water Bottle - 30 oz. — 2339 transactions
2. Patch Kit/8 Pat...
18:34:20 | INFO     | chart:  bar
18:34:20 | INFO     | duration: 19.42s
18:34:23 | INFO     |

AgentAnswer(answer='Top 5 products by transaction count:\n1. Water Bottle - 30 oz. — 2339 transactions\n2. Patch Kit/8 Patches — 1679 transactions\n3. Mountain Tire Tube — 1579 transactions\n4. AWC Logo Cap — 1324 transactions\n5. Sport-100 Helmet, Red — 1260 transactions', sql_used='SELECT p.Name, COUNT(th.TransactionID) AS TransactionCount\nFROM TransactionHistory th\nJOIN Product p ON th.ProductID = p.ProductID\nGROUP BY p.Name\nORDER BY TransactionCount DESC\nLIMIT 5;', chart_type='bar', plotly_code='import plotly.express as px\nfig = px.bar(df, x="Name", y="TransactionCount", title="Top 5 Products by Transaction Count")\nfig.update_layout(xaxis_title="Product Name", yaxis_title="Transaction Count", xaxis_tickangle=-45)')

In [118]:
conn.close()

In [5]:
import subprocess
import os
import signal

def clear_duckdb_lock(db_name="AdventureWorks.duckdb"):
    """Finds the PID locking the database and kills it."""
    try:
        # 1. Run 'lsof -t' to get ONLY the PID of the process using the file
        # -t is 'terse' mode, making it much easier to pipe into kill
        cmd = f"lsof -t {db_name}"
        pid_bytes = subprocess.check_output(cmd.split())

        # 2. Convert bytes to string and then to integer
        pid = int(pid_bytes.decode().strip())

        # 3. Kill the process
        print(f"Found lock on {db_name} by PID {pid}. Killing...")
        os.kill(pid, signal.SIGKILL)
        print("Lock cleared.")

    except subprocess.CalledProcessError:
        # This happens if lsof finds nothing (no lock exists)
        print(f"No active locks found on {db_name}.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Usage:
clear_duckdb_lock()

No active locks found on AdventureWorks.duckdb.


lsof: status error on AdventureWorks.duckdb: No such file or directory
lsof 4.91
 latest revision: ftp://lsof.itap.purdue.edu/pub/tools/unix/lsof/
 latest FAQ: ftp://lsof.itap.purdue.edu/pub/tools/unix/lsof/FAQ
 latest man page: ftp://lsof.itap.purdue.edu/pub/tools/unix/lsof/lsof_man
 usage: [-?abhlnNoOPRtUvVX] [+|-c c] [+|-d s] [+D D] [+|-f[cgG]]
 [-F [f]] [-g [s]] [-i [i]] [+|-L [l]] [+|-M] [-o [o]] [-p s]
 [+|-r [t]] [-s [p:s]] [-S [t]] [-T [t]] [-u s] [+|-w] [-x [fl]] [--] [names]
Use the ``-h'' option to get more help information.
